In [1]:
# Step 2: Feature Engineering V8p
# Run this notebook from BattingEdge_FYP/notebooks

import os, json
from pathlib import Path
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import joblib
from collections import defaultdict

# Paths
ROOT = Path("..")
SRC_ROOT = ROOT / "data" / "dataset_v8_balanced_videos"
OUT_ROOT = ROOT / "data" / "features" / "dataset_v8p"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
CLASSES = ["drive", "pull", "sweep", "cut"]

SEQUENCE_LENGTH = 50
FEATURE_COUNT = 99   # skeleton coords
BIOMECH_DIM = 4      # wrist velocity, knee angle, bat angle, stance width

# Save label classes for consistency
label_classes_path = ROOT / "backend" / "models" / "label_classes.json"
label_classes_path.parent.mkdir(parents=True, exist_ok=True)
with open(label_classes_path, "w") as f:
    json.dump(CLASSES, f)

# Mediapipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=2,
                    enable_segmentation=False, min_detection_confidence=0.55,
                    min_tracking_confidence=0.55)

# landmark indices
LW, RW = 15, 16
LS, RS = 11, 12
LK, RK = 25, 26
LA, RA = 27, 28

def time_resample(features, target_len=SEQUENCE_LENGTH):
    """Resample sequence to fixed length using interpolation."""
    T, D = features.shape
    if T == target_len: return features
    if T == 1: return np.tile(features[0], (target_len, 1))
    src_idx = np.arange(T)
    dst_idx = np.linspace(0, T-1, target_len)
    out = np.zeros((target_len, D), dtype=np.float32)
    for d in range(D):
        out[:, d] = np.interp(dst_idx, src_idx, features[:, d])
    return out

def angle_deg(a, b, c):
    ax, ay = a; bx, by = b; cx, cy = c
    v1 = np.array([ax-bx, ay-by]); v2 = np.array([cx-bx, cy-by])
    n1 = np.linalg.norm(v1)+1e-6; n2 = np.linalg.norm(v2)+1e-6
    cosang = np.clip(np.dot(v1, v2)/(n1*n2), -1.0, 1.0)
    return np.degrees(np.arccos(cosang))

def bat_angle_deg(lw, rw, ls, rs):
    v1 = np.array([rw[0]-lw[0], rw[1]-lw[1]])
    v2 = np.array([rs[0]-ls[0], rs[1]-ls[1]])
    n1 = np.linalg.norm(v1)+1e-6; n2 = np.linalg.norm(v2)+1e-6
    cosang = np.clip(np.dot(v1, v2)/(n1*n2), -1.0, 1.0)
    return np.degrees(np.arccos(cosang))

def stance_width(la, ra):
    return float(np.linalg.norm(np.array(ra) - np.array(la)))

def process_video(path: Path):
    """Extract skeleton + biomechanical features from a video."""
    cap = cv2.VideoCapture(str(path))
    frames_raw = []
    wrists_seq = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        res = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if not res.pose_landmarks: continue
        lms = res.pose_landmarks.landmark

        # 99 features (x,y,z for 33 landmarks)
        vec = []
        for lm in lms:
            vec.extend([lm.x, lm.y, lm.z])
        frames_raw.append(vec)

        # biomech inputs
        lw = (lms[LW].x, lms[LW].y); rw = (lms[RW].x, lms[RW].y)
        ls = (lms[LS].x, lms[LS].y); rs = (lms[RS].x, lms[RS].y)
        lk = (lms[LK].x, lms[LK].y); rk = (lms[RK].x, lms[RK].y)
        la = (lms[LA].x, lms[LA].y); ra = (lms[RA].x, lms[RA].y)
        wrists_seq.append((lw, rw, ls, rs, lk, rk, la, ra))
    cap.release()

    if len(frames_raw) == 0:
        return None

    X = np.array(frames_raw, dtype=np.float32)  # (T, 99)

    # biomech per-frame
    wrist_vel = [0.0]
    knee_angle = [180.0]
    bat_angle = [90.0]
    stance = [0.2]
    for t in range(1, len(wrists_seq)):
        lw_prev, rw_prev, *_ = wrists_seq[t-1]
        lw, rw, ls, rs, lk, rk, la, ra = wrists_seq[t]
        v_l = np.linalg.norm(np.array(lw) - np.array(lw_prev))
        v_r = np.linalg.norm(np.array(rw) - np.array(rw_prev))
        wrist_vel.append(float(max(v_l, v_r)))
        knee_angle.append(angle_deg(lk, lk, la))
        bat_angle.append(bat_angle_deg(lw, rw, ls, rs))
        stance.append(stance_width(la, ra))

    biomech = np.stack([wrist_vel, knee_angle, bat_angle, stance], axis=1)  # (T, 4)
    X_plus = np.concatenate([X, biomech], axis=1)  # (T, 103)

    # resample to fixed 50 frames
    X_50 = time_resample(X_plus, SEQUENCE_LENGTH)
    return X_50

# === Main loop with progress bars ===
counts = defaultdict(lambda: defaultdict(int))
train_feature_list = []

for split in SPLITS:
    out_split = OUT_ROOT / split
    out_split.mkdir(parents=True, exist_ok=True)

    for c in CLASSES:
        src_dir = SRC_ROOT / split / c
        out_dir = out_split / c
        out_dir.mkdir(parents=True, exist_ok=True)
        vids = list(src_dir.rglob("*.mp4")) + list(src_dir.rglob("*.avi")) + list(src_dir.rglob("*.mov"))

        print(f"\nProcessing {split}/{c} ({len(vids)} videos)...")
        for v in tqdm(vids, desc=f"{split}/{c}", unit="video"):
            try:
                X = process_video(v)
                if X is None:
                    continue
                np.save(out_dir / f"{v.stem}.npy", X)
                counts[split][c] += 1
                if split == "train":
                    train_feature_list.append(X)
            except Exception as e:
                print(f"⚠️ Error processing {v.name}: {e}")

# === Summary ===
print("\nFeature counts (videos successfully processed):")
for split in SPLITS:
    total = sum(counts[split].values())
    print(f"{split.upper()} total: {total}")
    for c in CLASSES:
        print(f"  {c}: {counts[split][c]}")

# === Fit scaler on train frames ===
if train_feature_list:
    X_train_concat = np.concatenate(train_feature_list, axis=0)  # (N_frames, 103)
    scaler = StandardScaler()
    scaler.fit(X_train_concat)
    scaler_path = ROOT / "backend" / "models" / "shot_scaler_V8p.pkl"
    scaler_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(scaler, scaler_path)
    print("\nScaler saved:", scaler_path)
else:
    print("No train features found; scaler not saved.")



Processing train/drive (381 videos)...


train/drive:   0%|          | 0/381 [00:00<?, ?video/s]d:\Users\Anoshia\BattingEdge_FYP\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
train/drive: 100%|██████████| 381/381 [1:19:41<00:00, 12.55s/video]



Processing train/pull (380 videos)...


train/pull: 100%|██████████| 380/380 [1:06:49<00:00, 10.55s/video]



Processing train/sweep (380 videos)...


...


Processing test/sweep (25 videos)...


test/sweep: 100%|██████████| 25/25 [04:31<00:00, 10.85s/video]



Processing test/cut (61 videos)...


test/cut: 100%|██████████| 61/61 [11:33<00:00, 11.36s/video]



Feature counts (videos successfully processed):
TRAIN total: 1520
  drive: 380
  pull: 380
  sweep: 380
  cut: 380
VAL total: 150
  drive: 45
  pull: 47
  sweep: 23
  cut: 35
TEST total: 219
  drive: 73
  pull: 60
  sweep: 25
  cut: 61

Scaler saved: ..\backend\models\shot_scaler_V8p.pkl
